In [5]:
import sys
import os
import copy

# Compute absolute path to the src/ folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")

# Add src/ to sys.path
sys.path.insert(0, src_path)

# Now you can import the code
from qvarnet.hamiltonians import GeneralHamiltonian, HarmonicOscillator
from qvarnet.models.mlp import MLP
from qvarnet.samplers import MetropolisHastingsSampler


import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from qvarnet.utils.callback import EarlyStoppingCallback 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## PARAMETERS

In [ ]:
EPOCHS             = 1_000
N_SAMPLES          = 10_000
L_BOX              = 20.0
STEP_SIZE          = 0.1
BURN_IN            = 500
CALLBACK_PATIENCE  = 10000
CALLBACK_MIN_DELTA = 1e-5
LEARNING_RATE      = 1e-3
MLP_LAYER_DIMS     = [1, 10, 1]


## Callback definition

In [7]:
callback = EarlyStoppingCallback(patience=10, min_delta=1e-4) 

## Model definition

In [9]:
mlp = MLP(layer_dims=[1,30,1])

## Hamiltonian definition

In [11]:
hamiltonian = HarmonicOscillator(model=mlp)

## Optimizer definition

In [ ]:
optimizer = torch.optim.Adam(mlp.parameters(), lr=LEARNING_RATE)

In [ ]:

#----------------------- DEFINE PARAMETERS -----------------------
EPOCHS             = 1_000
N_SAMPLES          = 10_000
L_BOX              = 20.0
STEP_SIZE          = 0.1
BURN_IN            = 500
CALLBACK_PATIENCE  = 10000
CALLBACK_MIN_DELTA = 1e-5
LEARNING_RATE      = 1e-3
MLP_LAYER_DIMS     = [1, 10, 1]
DEBUG              = False

wf_history      = []
samples_history = []
energy_history  = []

local_energy_history = []
samples_history = []
dict_state_history = []

#----------------------- DEFINE MODEL TOPOLOGY -----------------------
model = MLP(layer_dims=MLP_LAYER_DIMS)
model.to(device)

model_for_trapz = copy.deepcopy(model)  # Create a copy of the model for trapezoidal integration

# init weights

def init_weights(m):
    if isinstance(m, nn.Linear):
        # Small random weights - very important for VMC!
        nn.init.normal_(m.weight, mean=0.0, std=0.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


model.apply(init_weights)

#------------------------ DEFINE HAMILTONIAN -----------------------
hamiltonian = HarmonicOscillator(model=model)
hamiltonian.to(device)

trapezoid_hamiltonian = HarmonicOscillator(model=model_for_trapz)
trapezoid_hamiltonian.to(device)

#------------------------ DEFINE SAMPLER -----------------------
sampler = MetropolisHastingsSampler(
    model     = model,
    n_samples = N_SAMPLES,
    step_size = STEP_SIZE,
    burn_in   = BURN_IN,
    is_wf     = True,
    L_BOX     = L_BOX,
)
sampler.to(device)

# callback = EarlyStoppingCallback(patience=CALLBACK_PATIENCE, min_delta=CALLBACK_MIN_DELTA)
callback = None  # Disable early stopping for now

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

sampling_times = []
energy_times = []

x_trapezoid = torch.linspace(-L_BOX/2, L_BOX/2, 10_000).view(-1, 1).to(device)
x_trapezoid.requires_grad = True  # Ensure x_trapezoid is differentiable

for epoch in tqdm(range(EPOCHS)):
    optimizer.zero_grad()
    
    # x0 = torch.randn(1, device=device)
    start_sample_time = time.time()
    # Run sampler
    sampler.model = model
    x0 = torch.tensor([0.0], device=device)  # Initial point for the sampler
    samples = sampler(x0, method="parallel", n_walkers=(N_SAMPLES)).requires_grad_(False)
    # samples.requires_grad = True
    end_sample_time = time.time()
    sampling_times.append(end_sample_time - start_sample_time)
    samples_history.append(samples.cpu().detach())
    dict_state_history.append(model.state_dict())
    
    # Compute the mean and std of the local energy
    start_energy_time = time.time()
    hamiltonian.model = model
    local_energy = hamiltonian(samples)
    squared_psi = model(samples).pow(2)
    sampled_norm = torch.mean(squared_psi)
    local_energy_history.append(local_energy.cpu().detach())
    
    psi_values = model(samples)
    psi_std = psi_values.std()
    
    if psi_std < 1e-4:
        print(f"WARNING: Wavefunction becoming flat at epoch {epoch}! Restarting with new initialization...")
        model.apply(init_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        continue
    
    psi_trapezoid = model(x_trapezoid)
    local_energy_trapz = trapezoid_hamiltonian(x_trapezoid)
    numerator = torch.trapz(local_energy_trapz * psi_trapezoid**2, x_trapezoid, dim=0)
    denominator = torch.trapz(psi_trapezoid**2, x_trapezoid, dim=0)
    energy_trapzs = numerator / denominator
    loss_trapzs = energy_trapzs
    
    psi_trapezoid = psi_trapezoid / torch.sqrt(denominator)  # Normalize the wavefunction
    
    end_energy_time = time.time()
    energy_times.append(end_energy_time - start_energy_time)
    
    # if EPOCHS == 1:
    #     plt.hist(samples.cpu().detach().numpy(), bins=100, density=True, alpha=0.7, label='Sampled')
    #     plt.plot(x_trapezoid.cpu().detach().numpy(), psi_trapezoid.cpu().detach().numpy()**2, label='Trapezoid Wavefunction')
    # elif epoch % 100 == 0:
    #     plt.hist(samples.cpu().detach().numpy(), bins=100, density=True, alpha=0.7, label='Sampled')
    #     plt.plot(x_trapezoid.cpu().detach().numpy(), psi_trapezoid.cpu().detach().numpy()**2, label='Trapezoid Wavefunction')
    #     plt.show()

    loss = local_energy.mean()
    
    # dot = make_dot(loss, params=dict(model.named_parameters()))
    # dot.render("loss_graph", format="png")  # Saves as loss_graph.png
    # dot.view()  # Opens the graph

    # print(f"ENERGY MEAN: {loss.item():.2f}±{local_energy.std().item():.2f}")

    if callback is not None:
        callback(epoch, energy, model)
        if callback.stop_training:
            print("Early stopping triggered.")
            break
    
    wf_history.append(model.state_dict())
    samples_history.append(samples)

    
    # Compute gradients
    loss.squeeze().backward()
    
    # loss_trapzs.squeeze().backward()
    
    # # PRINT GRADIENTS FOR THE TWO METHODS
    if epoch % 100 == 0:
        print("GRADIENTS MC:")
        for name, param in model.named_parameters():
            if param is not None:
                print(f"Parameter {name}:\n {param.grad.data.cpu().numpy() if param.grad is not None else None}")

    # print("GRADIENTS TRAPEZOID:")
    # for name, param in model_for_trapz.named_parameters():
    #     if param is not None:
    #         print(f"Parameter {name}:\n {param.grad.data.cpu().numpy() if param.grad is not None else None}")
    # Update model parameters
    optimizer.step()
    # trapezoid_optimizer.step()
    
    sampler.reset_statistics()

print("Ended training.")